# VXN robustness — is the NASDAQ win real? (IXIC only, h=5, embargo applied)

Swaps the S&P-based VIX for the NASDAQ-specific VXN as the implied-vol input/core predictor, on
IXIC only, to check whether the IXIC-specific hybrid win is an artifact of using the "wrong" (S&P)
implied-vol series for a NASDAQ index. Same embargoed splits as every other notebook here.

Approximate runtime: **~15 minutes** (2 cores x 3 seeds = 6 trainings).


# VXN Robustness Test — is the NASDAQ win real?

The 3-index run found the hybrid's clearest, statistically significant win on **IXIC** — and our story
was *"VIX is the S&P's implied vol, a poor proxy for NASDAQ, so the deep correction has room."*
The obvious referee rebuttal: *"then use **VXN** (NASDAQ's own implied vol) instead of VIX."*

This notebook answers that directly, on IXIC only:
- **Core A (VIX):** frozen OLS on HAR-X + **VIX** + GARCH → hybrid.
- **Core B (VXN):** frozen OLS on HAR-X + **VXN** + GARCH → hybrid.
- Diebold–Mariano on daily QLIKE for the decisive pairs.

**Reading the outcome:** if `Hybrid (VXN core)` still significantly beats `VXN-only` / `core OLS (VXN)`,
the claim is bulletproof. If instead `VXN-only` closes the gap and the hybrid adds nothing, the IXIC win
was mostly "the baseline was handed the wrong implied vol" — a finding we would report honestly.

**Data:** regenerate the CSVs with the updated `build_vol_dataset.py` (it now also pulls `^VXN`), then
place `IXIC_vol.csv` (and the other two, optional here) next to this notebook. Requires the new columns
`VXN`, `VXN_chg` in the CSV.

## 0 — Setup
Put `IXIC_vol.csv`, `NYA_vol.csv`, `DJI_vol.csv` (from `build_vol_dataset.py`) in the working directory.

In [1]:
import os
EXPECTED = ["IXIC_vol.csv", "NYA_vol.csv", "DJI_vol.csv"]
missing = [f for f in EXPECTED if not os.path.exists(f)]
print("found:", [f for f in EXPECTED if os.path.exists(f)])
if missing:
    print("MISSING:", missing, "\n-> upload them, or run build_vol_dataset.py first (needs internet):")
    print("   !pip install -q yfinance && python build_vol_dataset.py  (then move vol_dataset/*.csv here)")

found: ['IXIC_vol.csv', 'NYA_vol.csv', 'DJI_vol.csv']


In [2]:
import math, random, copy
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch", torch.__version__)
if DEVICE == "cuda": print("gpu:", torch.cuda.get_device_name(0))

## 1 — Config
Every experiment is one `Config`. All ablation knobs below are implemented.

In [3]:
INDEX = "IXIC"                          # this robustness test runs on NASDAQ only

@dataclass
class Config:
    data_path: str = "IXIC_vol.csv"
    target_col: str = "target_logrv"                    # log realized vol over the next `target_horizon` days
    not_features: List[str] = field(default_factory=lambda: ["Close", "target_logrv", "target_rv"])
    target_horizon: int = 5                             # h used to BUILD the target/baselines (must match build_vol_dataset.py)
    iv_col: str = "VIX"                                 # which implied-vol column feeds the core & the IV-only baseline ("VIX" | "VXN")

    seq_in: int = 60                                    # long lookback: volatility has long memory
    horizon: int = 1                                    # model output dim (one target per sample)
    val_ratio: float = 0.15; test_ratio: float = 0.15

    d_model: int = 32; n_layer: int = 3; d_state: int = 128; expand: int = 2; d_conv: int = 3
    dt_rank: Optional[int] = None                       # None -> ceil(d_model/16)
    cheb_k: int = 3; embed: int = 10; hid: int = 32
    scan_chunk: int = 32                                # parallel-scan chunk (>= seq_len => single fully-parallel scan)

    batch_size: int = 32; epochs: int = 100; lr: float = 1e-3; weight_decay: float = 1e-4
    grad_clip: float = 5.0; patience: int = 15

    # ----- ablation knobs (pure BI-Mamba) -----
    scan_axis: str = "time"                             # "time" (baseline) | "feature" (variate-as-token)
    n_scans: int = 2                                    # 2 = bidirectional; >2 adds permuted scans
    use_registers: bool = False; n_registers: int = 4
    reg_control: bool = False                           # param-matched control for the register readout
    grad_checkpoint: bool = False                       # recompute scan in backward -> big GPU-memory saving on heavy rows
    hybrid: bool = False                                # add the HAR-X+VIX+GARCH core; BI-Mamba models the residual
    freeze_core: bool = True                            # lock the core to its OLS fit (two-stage) so you can't do worse than it
    qlike_align: bool = False                           # train & early-stop on QLIKE (variance loss) instead of MSE/RMSE
    cross_market: bool = False                          # input = features of ALL three indices (spillover); target stays this index
    cross_paths: List[str] = field(default_factory=lambda: ["DJI_vol.csv", "IXIC_vol.csv", "NYA_vol.csv"])

    seed: int = 1; device: str = DEVICE
    def __post_init__(self):
        if self.dt_rank is None: self.dt_rank = math.ceil(self.d_model / 16)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def replace(cfg, **kw):
    d = asdict(cfg); d.update(kw); return Config(**d)

## 2 — Data pipeline (clean)

Read the per-index CSV, drop the non-feature columns (`Close`, `target_rv`), use `target_logrv` as
the label. Split chronologically, **standardize** features and target on **train only** (z-score is
right for log-RV and robust to the COVID spike that min-max would crush), then window. Each window's
label is `target_logrv` at its **last input day** — which already looks `h` days ahead, so the target
window never overlaps the inputs (leak-free by construction; the builder also self-checks this).

In [4]:
import warnings; warnings.filterwarnings("ignore")
from arch import arch_model

# ---- rolling GARCH(1,1) h-day vol forecast (causal: only past returns); cached so it runs once ----
def _garch_hday_logvol(r, h, refit=25, min_train=500):
    r = np.asarray(r, float); n = len(r); out = np.full(n, np.nan)
    mu = om = al = be = sbar = cv = None; last = -10**9
    for t in range(n):
        if t >= min_train and (t - last) >= refit:
            res = arch_model(pd.Series(r[:t+1]), mean="Constant", vol="Garch", p=1, q=1, dist="normal").fit(disp="off")
            pa = res.params; mu, om, al, be = pa["mu"], pa["omega"], pa["alpha[1]"], pa["beta[1]"]
            ps = al + be; sbar = om / (1 - ps) if 0 < ps < 1 else float(np.var(r[:t+1]))
            cv = float(np.asarray(res.conditional_volatility)[-1] ** 2); last = t
        if cv is None: continue
        s_next = om + al * (r[t] - mu) ** 2 + be * cv          # sigma^2_{t+1}
        ps = al + be; ksum = 0.0; ek = s_next
        for _ in range(h): ksum += ek; ek = sbar + ps * (ek - sbar)   # sum E[sigma^2_{t+1..t+h}]
        out[t] = 0.5 * np.log(ksum) - np.log(100)              # log h-day vol (decimal)
        cv = s_next
    return out

_GARCH_CACHE = {}
def garch_series(path, h, refit=25, min_train=500):
    key = (path, h, refit, min_train)
    if key not in _GARCH_CACHE:
        df = pd.read_csv(path, index_col=0, parse_dates=True).sort_index()
        r_full = (np.log(df["Close"].astype("float64")).diff() * 100).values      # NaN at [0]
        valid = np.isfinite(r_full)
        g = _garch_hday_logvol(r_full[valid], h, refit, min_train)                # GARCH on finite returns
        out = np.full(len(r_full), np.nan); out[valid] = g
        _GARCH_CACHE[key] = pd.Series(out, index=df.index)
    return _GARCH_CACHE[key]

class Standardizer:                                     # per-feature z-score (robust to outliers vs min-max)
    def fit(self, x):
        self.mu = x.mean(0); self.sd = x.std(0)
        self.sd = np.where(self.sd == 0, 1.0, self.sd); return self
    def transform(self, x): return (x - self.mu) / self.sd

class Standardizer1D:
    def fit(self, x):
        self.mu = float(np.mean(x)); self.sd = float(np.std(x)) or 1.0; return self
    def transform(self, x): return (x - self.mu) / self.sd
    def inverse(self, x):   return x * self.sd + self.mu

def _core_frame(df, cfg):
    close = df["Close"].astype("float64"); var = (np.log(close).diff()) ** 2; eps = 1e-12
    core = pd.DataFrame(index=df.index)
    core["hd"] = np.log(np.sqrt(var.rolling(1).sum()) + eps)            # HAR daily
    core["hw"] = np.log(np.sqrt(var.rolling(5).sum()) + eps)            # HAR weekly
    core["hm"] = np.log(np.sqrt(var.rolling(22).sum()) + eps)           # HAR monthly
    if cfg.iv_col not in df.columns:
        raise ValueError(f"column {cfg.iv_col!r} not in {cfg.data_path} — regenerate the CSVs with the updated build_vol_dataset.py (adds ^VXN)")
    core["iv"] = np.log(df[cfg.iv_col].astype("float64") + eps)
    core["garch"] = garch_series(cfg.data_path, cfg.target_horizon).reindex(df.index)   # GARCH(1,1)
    return core

def prepare_data(cfg):
    df = pd.read_csv(cfg.data_path, index_col=0, parse_dates=True).sort_index()   # chronological
    y = df[cfg.target_col].astype("float32")                                      # target_logrv (causal, future)
    feats = df.drop(columns=[c for c in cfg.not_features if c in df.columns]).astype("float32")
    if cfg.cross_market:                                                          # add the OTHER indices' full feature sets
        for p in [q for q in cfg.cross_paths if q != cfg.data_path]:
            od = pd.read_csv(p, index_col=0, parse_dates=True).sort_index()
            of = od.drop(columns=[c for c in cfg.not_features if c in od.columns]).astype("float32")
            of.columns = [f"{p.split('_')[0]}:{c}" for c in of.columns]
            feats = feats.join(of, how="inner")                                   # shared calendar (same builder)
        df, y = df.loc[feats.index], y.loc[feats.index]
    core = _core_frame(df, cfg)                                                    # HAR-X + VIX + GARCH (this index only)

    m = feats.notna().all(1) & y.notna() & core.notna().all(1)                    # drop warmup/no-future rows
    feats, y, core = feats[m].values, y[m].values, core[m].values.astype("float32")
    N, Fdim = feats.shape; n_core = core.shape[1]

    n_test = int(N * cfg.test_ratio); n_val = int(N * cfg.val_ratio); n_train = N - n_val - n_test
    tr_end, val_end = n_train, n_train + n_val

    fscaler = Standardizer().fit(feats[:tr_end]); cscaler = Standardizer().fit(core[:tr_end])  # fit on TRAIN
    tscaler = Standardizer1D().fit(y[:tr_end])
    Xs = fscaler.transform(feats).astype("float32"); Cs = cscaler.transform(core).astype("float32")
    ys = tscaler.transform(y).astype("float32")

    # frozen-core OLS (standardized core -> standardized target), fit on TRAIN -> the hybrid's locked baseline
    A = np.c_[np.ones(tr_end), Cs[:tr_end]]; coef, *_ = np.linalg.lstsq(A, ys[:tr_end], rcond=None)
    core_coef = (coef[1:].astype("float32"), float(coef[0]))                       # (weights[n_core], bias)

    Xw, Cw, Yw, tgt = [], [], [], []
    for i in range(N - cfg.seq_in + 1):
        last = i + cfg.seq_in - 1                                                 # label/core at the LAST input day
        Xw.append(Xs[i:i+cfg.seq_in]); Cw.append(Cs[last]); Yw.append([ys[last]]); tgt.append(last)
    Xw = np.asarray(Xw, "float32"); Cw = np.asarray(Cw, "float32")
    Yw = np.asarray(Yw, "float32")[..., None]; tgt = np.asarray(tgt)

    h = cfg.target_horizon                                                        # embargo depth: forward target window
    def split(lo, hi):
        s = (tgt >= lo) & (tgt < hi)
        return s, torch.from_numpy(Xw[s]), torch.from_numpy(Cw[s]), torch.from_numpy(Yw[s])
    s_tr, Xtr, Ctr, Ytr = split(0, tr_end - h)          # purge last h train samples (target reaches into val)
    s_va, Xva, Cva, Yva = split(tr_end, val_end - h)    # purge last h val samples (target reaches into test)
    s_te, Xte, Cte, Yte = split(val_end, N)             # test needs no purge -- nothing follows it

    if s_tr.any() and s_va.any():
        assert tgt[s_tr].max() + h < tgt[s_va].min(), "train/val target-window overlap detected!"
    if s_va.any() and s_te.any():
        assert tgt[s_va].max() + h < tgt[s_te].min(), "val/test target-window overlap detected!"
    n_drop_tr = int(((tgt >= tr_end - h) & (tgt < tr_end)).sum())
    n_drop_va = int(((tgt >= val_end - h) & (tgt < val_end)).sum())
    print(f"purge[{cfg.data_path}, h={h}]: dropped {n_drop_tr} train-boundary + {n_drop_va} val-boundary rows | "
          f"train={int(s_tr.sum())} val={int(s_va.sum())} test={int(s_te.sum())} -- zero target-window overlap OK")

    return {"train": (Xtr, Ctr, Ytr), "val": (Xva, Cva, Yva), "test": (Xte, Cte, Yte)}, Fdim, tscaler, n_core, core_coef

def loaders(data, bs):
    return {k: torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X, C, Y),
            batch_size=bs, shuffle=(k == "train"), drop_last=False) for k, (X, C, Y) in data.items()}

In [5]:
data, NF, ts, NCORE, CORE_COEF = prepare_data(Config())
for k, (X, C, Y) in data.items(): print(f"{k:5s} X={tuple(X.shape)}  core={tuple(C.shape)}  Y={tuple(Y.shape)}")
print("n_features:", NF, "| n_core (HAR-X+VIX+GARCH):", NCORE)

purge[IXIC_vol.csv, h=5]: dropped 5 train-boundary + 5 val-boundary rows | train=2486 val=541 test=546 -- zero target-window overlap OK
train X=(2486, 60, 39)  core=(2486, 5)  Y=(2486, 1, 1)
val   X=(541, 60, 39)  core=(541, 5)  Y=(541, 1, 1)
test  X=(546, 60, 39)  core=(546, 5)  Y=(546, 1, 1)
n_features: 39 | n_core (HAR-X+VIX+GARCH): 5


## 3 — Model (pure BI-Mamba + optional hybrid core)

Backbone = **bidirectional Mamba** (chunked parallel scan) with the toggles built in; a mean-pool +
linear **readout** replaces the graph. `scan_axis="feature"` makes the features the sequence
(variate-as-token); `n_scans` sums that many permuted scan directions; `use_registers` inserts
evenly-spaced register tokens with a reduce-&-concat readout (`reg_control` = same params, no
registers). With `hybrid=True`, output = `linear(HAR_d, HAR_w, HAR_m, VIX) + BiMamba_correction`, and
the correction is zero-initialised so training starts exactly at the linear baseline.

In [6]:
from torch.utils.checkpoint import checkpoint

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x): return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

class MambaBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__(); self.cfg = cfg; di = cfg.expand * cfg.d_model
        self.in_proj = nn.Linear(cfg.d_model, di * 2, bias=False)
        self.conv1d = nn.Conv1d(di, di, cfg.d_conv, groups=di, padding=cfg.d_conv - 1, bias=True)
        self.x_proj = nn.Linear(di, cfg.dt_rank + cfg.d_state * 2, bias=False)
        self.dt_proj = nn.Linear(cfg.dt_rank, di, bias=True)
        A = torch.arange(1, cfg.d_state + 1, dtype=torch.float32).repeat(di, 1)
        self.A_log = nn.Parameter(torch.log(A)); self.D = nn.Parameter(torch.ones(di))
        self.out_proj = nn.Linear(di, cfg.d_model, bias=False)
    def forward(self, x):
        b, l, _ = x.shape
        x_, res = self.in_proj(x).chunk(2, -1)
        x_ = F.silu(self.conv1d(x_.transpose(1, 2))[..., :l].transpose(1, 2))
        return self.out_proj(self._ssm(x_) * F.silu(res))
    def _ssm(self, x):
        A = -torch.exp(self.A_log.float()); D = self.D.float()
        delta, B, C = self.x_proj(x).split([self.cfg.dt_rank, self.cfg.d_state, self.cfg.d_state], -1)
        delta = F.softplus(self.dt_proj(delta))
        dA  = torch.exp(torch.einsum('bld,dn->bldn', delta, A))
        dBu = torch.einsum('bld,bln,bld->bldn', delta, B, x)
        return self._selective_scan(dA, dBu, C) + x * D
    def _selective_scan(self, dA, dBu, C):
        # chunked parallel scan: parallel within chunks, sequential across, n contracted per chunk
        L = dA.shape[1]; chunk = min(self.cfg.scan_chunk, L); carry, ys = None, []
        for s in range(0, L, chunk):
            a, bx, c = dA[:, s:s+chunk], dBu[:, s:s+chunk], C[:, s:s+chunk]
            h = self._chunk_scan(a, bx)
            if carry is not None: h = h + torch.cumprod(a, 1) * carry.unsqueeze(1)
            ys.append(torch.einsum('bcdn,bcn->bcd', h, c)); carry = h[:, -1]
        return torch.cat(ys, 1)
    @staticmethod
    def _chunk_scan(a, bx):                              # Hillis-Steele inclusive scan (zero init)
        L = a.shape[1]; shift = 1
        while shift < L:
            ap = F.pad(a,  (0, 0, 0, 0, shift, 0), value=1.0)[:, :L]
            bp = F.pad(bx, (0, 0, 0, 0, shift, 0), value=0.0)[:, :L]
            bx = a * bp + bx; a = a * ap; shift *= 2
        return bx

class ResidualBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__(); self.norm = nn.LayerNorm(cfg.d_model); self.mixer = MambaBlock(cfg)
    def forward(self, x): return self.mixer(self.norm(x))

class BiMamba(nn.Module):
    def __init__(self, cfg, nf):
        super().__init__(); self.cfg = cfg
        self.seq_len, self.token_in = (cfg.seq_in, nf) if cfg.scan_axis == "time" else (nf, cfg.seq_in)
        self.embedding = nn.Linear(self.token_in, cfg.d_model)
        self.ns = cfg.n_scans
        self.scans = nn.ModuleList([nn.ModuleList([ResidualBlock(cfg) for _ in range(cfg.n_layer)])
                                    for _ in range(self.ns)])
        self.use_reg = cfg.use_registers
        self.L_ext = self.seq_len + (cfg.n_registers if self.use_reg else 0)
        m = lambda nrm: nn.Sequential(nrm, nn.Linear(self.L_ext, cfg.hid), nn.ReLU(), nn.Linear(cfg.hid, self.L_ext))
        self.lin = nn.ModuleList([m(nn.LayerNorm(self.L_ext))] + [m(RMSNorm(self.L_ext)) for _ in range(cfg.n_layer - 1)])
        self.norm_f = nn.LayerNorm(cfg.d_model); self.head = nn.Linear(cfg.d_model, self.token_in)
        g = torch.Generator().manual_seed(cfg.seed)
        perms = [torch.arange(self.L_ext), torch.arange(self.L_ext).flip(0)]
        for _ in range(self.ns - 2): perms.append(torch.randperm(self.L_ext, generator=g))
        for i, p in enumerate(perms[:self.ns]):
            self.register_buffer(f"perm_{i}", p); self.register_buffer(f"inv_{i}", torch.argsort(p))
        self.global_dim = 0
        if self.use_reg:
            self.registers = nn.Parameter(torch.randn(cfg.n_registers, cfg.d_model) * 0.02)
            pos = torch.linspace(0, self.L_ext - 1, cfg.n_registers).round().long()
            assert len(torch.unique(pos)) == cfg.n_registers, "n_registers too large for seq_len"
            mask = torch.zeros(self.L_ext, dtype=torch.bool); mask[pos] = True
            self.register_buffer("reg_mask", mask)
            self.r = max(1, cfg.d_model // 4); self.reg_reduce = nn.Linear(cfg.d_model, self.r)
            self.global_dim = cfg.n_registers * self.r
        elif cfg.reg_control:
            self.r = max(1, cfg.d_model // 4); self.reg_reduce = nn.Linear(cfg.d_model, self.r)
            self.global_dim = cfg.n_registers * self.r
    def _insert(self, x):
        b, S, d = x.shape
        out = torch.empty(b, self.L_ext, d, device=x.device, dtype=x.dtype)
        out[:, self.reg_mask] = self.registers.to(x.dtype).expand(b, -1, -1)
        out[:, ~self.reg_mask] = x
        return out
    def _block(self, blk, x):
        if self.cfg.grad_checkpoint and self.training:
            return checkpoint(blk, x, use_reentrant=False)
        return blk(x)
    def forward(self, x):                               # x: [b, L, F]
        if self.cfg.scan_axis == "feature": x = x.transpose(1, 2)
        x = self.embedding(x)
        if self.use_reg: x = self._insert(x)
        for i in range(self.cfg.n_layer):
            agg = x
            for s in range(self.ns):
                p, inv = getattr(self, f"perm_{s}"), getattr(self, f"inv_{s}")
                agg = agg + self._block(self.scans[s][i], x[:, p])[:, inv]
            x = agg
            x = self.lin[i](x.transpose(1, 2)).transpose(1, 2) + x
        x = self.norm_f(x)
        gvec = None
        if self.use_reg:
            gvec = self.reg_reduce(x[:, self.reg_mask]).flatten(1)
            x = x[:, ~self.reg_mask]
        elif self.cfg.reg_control:
            gvec = self.reg_reduce(x).mean(1).repeat(1, self.cfg.n_registers)
        out = self.head(x)
        if self.cfg.scan_axis == "feature": out = out.transpose(1, 2)
        return out, gvec                                # [b, L, F], global vec or None

class BiMambaNet(nn.Module):
    # Pure bidirectional Mamba (graph removed). Optional HAR-X+VIX+GARCH core => hybrid.
    def __init__(self, cfg, nf, n_core=0, core_coef=None):
        super().__init__(); self.cfg = cfg; self.nf = nf
        self.backbone = BiMamba(cfg, nf)                              # BI-Mamba w/ scan/register knobs
        self.readout = nn.Linear(nf, cfg.horizon)                    # deep correction from pooled output
        self.hybrid = bool(cfg.hybrid and n_core > 0)
        if self.hybrid:
            self.core = nn.Linear(n_core, cfg.horizon)               # linear HAR-X+VIX+GARCH baseline
            nn.init.zeros_(self.readout.weight); nn.init.zeros_(self.readout.bias)   # start AT the baseline
            if cfg.freeze_core and core_coef is not None:            # two-stage: lock the core to its OLS fit
                w, b = core_coef
                with torch.no_grad():
                    self.core.weight.copy_(torch.as_tensor(w, dtype=torch.float32).view(1, -1))
                    self.core.bias.fill_(float(b))
                self.core.weight.requires_grad_(False); self.core.bias.requires_grad_(False)
        if self.backbone.global_dim:
            self.reg_head = nn.Linear(self.backbone.global_dim, cfg.horizon)
            if self.hybrid: nn.init.zeros_(self.reg_head.weight); nn.init.zeros_(self.reg_head.bias)
    def forward(self, x, core=None):
        h, gvec = self.backbone(x)                                   # h: [b, seq_in, nf]
        out = self.readout(h.mean(1))                               # [b, horizon]
        if gvec is not None: out = out + self.reg_head(gvec)
        if self.hybrid and core is not None: out = out + self.core(core)
        return out.unsqueeze(-1)                                     # [b, horizon, 1]

print("baseline params:", sum(p.numel() for p in BiMambaNet(Config(), NF, NCORE, CORE_COEF).parameters()))

baseline params: 203251


## 4 — Metrics (volatility)
On **log realized vol**: **RMSE**, **MAE** (↓); **R²** (Mincer–Zarnowitz, ↑); **Corr** (Pearson, ↑). On **variance** (`σ²=exp(2·logRV)`): **QLIKE** (↓), the standard robust volatility loss. Early stopping uses validation RMSE.

In [7]:
def _pearson(a, b):
    a, b = a - a.mean(), b - b.mean()
    den = a.std(unbiased=False) * b.std(unbiased=False)
    return torch.tensor(float("nan")) if den == 0 else (a * b).mean() / den

def _vol_metrics(pred, true):                              # pred/true = log realized vol
    err = pred - true
    rmse = (err ** 2).mean().sqrt().item(); mae = err.abs().mean().item()
    ss_res = (err ** 2).sum(); ss_tot = ((true - true.mean()) ** 2).sum()
    r2 = (1 - ss_res / ss_tot).item() if ss_tot > 0 else float("nan")
    vp, vt = torch.exp(2 * pred), torch.exp(2 * true)      # variance
    qlike = (vt / vp - torch.log(vt / vp) - 1).mean().item()
    return {"RMSE": rmse, "MAE": mae, "QLIKE": qlike, "R2": r2, "Corr": _pearson(pred, true).item()}

@torch.no_grad()
def evaluate(model, loader, tscaler, device, return_preds=False):
    model.eval(); ps, tsx = [], []
    for X, C, Y in loader:
        ps.append(model(X.to(device), C.to(device)).cpu().reshape(-1)); tsx.append(Y.reshape(-1))
    pred = torch.from_numpy(tscaler.inverse(torch.cat(ps).numpy()))   # predicted log-RV
    true = torch.from_numpy(tscaler.inverse(torch.cat(tsx).numpy()))  # actual log-RV
    m = _vol_metrics(pred, true)
    return (m, pred.numpy(), true.numpy()) if return_preds else m

## 5 — Train · multi-seed · ablation harness

In [8]:
def qlike_loss_std(pred_z, true_z, sd):        # differentiable QLIKE in standardized log-RV space
    r = torch.clamp((2.0 * sd) * (true_z - pred_z), -10.0, 10.0)
    return (torch.exp(r) - r - 1).mean()

def train_once(cfg, data, nf, tscaler, n_core, core_coef, verbose=False):
    set_seed(cfg.seed); dl = loaders(data, cfg.batch_size)
    model = BiMambaNet(cfg, nf, n_core, core_coef).to(cfg.device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sd = float(tscaler.sd); _mse = nn.MSELoss()
    lossf = (lambda p, y: qlike_loss_std(p, y, sd)) if cfg.qlike_align else (lambda p, y: _mse(p, y))
    es = "QLIKE" if cfg.qlike_align else "RMSE"        # early-stop metric aligned with the loss
    best, best_state, wait = float("inf"), None, 0
    for ep in range(cfg.epochs):
        model.train()
        for X, Cc, Y in dl["train"]:
            X, Cc, Y = X.to(cfg.device), Cc.to(cfg.device), Y.to(cfg.device)
            opt.zero_grad(); loss = lossf(model(X, Cc), Y); loss.backward()
            if cfg.grad_clip: nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            opt.step()
        vm = evaluate(model, dl["val"], tscaler, cfg.device)[es]
        if vm < best: best, best_state, wait = vm, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if cfg.patience and wait >= cfg.patience: break
        if verbose and ep % 10 == 0: print(f"  epoch {ep:3d} | val {es} {vm:.4f}")
    if best_state: model.load_state_dict(best_state)
    return evaluate(model, dl["test"], tscaler, cfg.device, return_preds=True)

def run_multiseed(cfg, seeds=(1, 2, 3), verbose=False):
    data, nf, ts, n_core, core_coef = prepare_data(cfg)
    mets, preds, true = [], [], None
    for s in seeds:
        m, p, t = train_once(replace(cfg, seed=s), data, nf, ts, n_core, core_coef, verbose)
        mets.append(m); preds.append(p); true = t
    ens = np.mean(np.stack(preds), 0)                  # seed-ensemble prediction (used for DM)
    stats = {k: (float(np.mean([r[k] for r in mets])), float(np.std([r[k] for r in mets]))) for k in mets[0]}
    return stats, ens, true

def vol_baselines(cfg, return_preds=False):
    import numpy.linalg as la
    df = pd.read_csv(cfg.data_path, index_col=0, parse_dates=True).sort_index()
    feats = df.drop(columns=[c for c in cfg.not_features if c in df.columns])
    y = df[cfg.target_col].astype("float64")
    close = df["Close"].astype("float64"); var = (np.log(close).diff()) ** 2
    h = cfg.target_horizon; eps = 1e-12
    persist = np.log(np.sqrt(var.rolling(h).sum()) + eps)
    ew = var.ewm(alpha=1 - 0.94, adjust=False).mean()
    ewma = 0.5 * np.log(ew * h + eps)
    hd = np.log(np.sqrt(var.rolling(1).sum()) + eps)
    hw = np.log(np.sqrt(var.rolling(5).sum()) + eps)
    hm = np.log(np.sqrt(var.rolling(22).sum()) + eps)
    garch = garch_series(cfg.data_path, h).reindex(df.index)
    cols = {"y": y, "persist": persist, "ewma": ewma, "hd": hd, "hw": hw, "hm": hm, "garch": garch}
    if cfg.iv_col in feats.columns: cols["ivlog"] = np.log(feats[cfg.iv_col].astype("float64") + eps)
    sub = pd.DataFrame(cols, index=df.index)
    core_cols = [c for c in ["hd", "hw", "hm", "ivlog", "garch"] if c in sub.columns]
    sub = sub[feats.notna().all(1) & y.notna() & sub[core_cols].notna().all(1)]   # same mask as the model

    N = len(sub); n_test = int(N * cfg.test_ratio); n_val = int(N * cfg.val_ratio); n_train = N - n_val - n_test
    tr = sub.iloc[:n_train].dropna(); te = sub.iloc[n_train + n_val:]
    tt = torch.tensor(te["y"].values)
    def ols(cols_):
        A = np.c_[np.ones(len(tr)), tr[cols_].values]; b, *_ = la.lstsq(A, tr["y"].values, rcond=None)
        return np.c_[np.ones(len(te)), te[cols_].values] @ b
    P = {"persistence (RW)": te["persist"].values, "EWMA (l=.94)": te["ewma"].values,
         "GARCH(1,1)": te["garch"].values, "HAR-RV": ols(["hd", "hw", "hm"])}
    if "ivlog" in sub.columns:
        P[f"{cfg.iv_col}-only"] = ols(["ivlog"])
        P[f"core OLS (HAR-X+{cfg.iv_col}+GARCH)"] = ols(core_cols)
    out = {n: _vol_metrics(torch.tensor(np.asarray(p)), tt) for n, p in P.items()}
    if return_preds: return out, {n: np.asarray(p, float) for n, p in P.items()}, te["y"].values.astype(float)
    return out

PREDS, TRUES = {}, {}                                   # tag -> {method: test predictions (log-RV)}, tag -> true
def run_ablation(named, seeds=(1, 2, 3), baselines=True, tag=None):
    res, preds, true_m = {}, {}, None
    for name, cfg in named.items():
        print(f"[run] {name}"); res[name], p, true_m = run_multiseed(cfg, seeds); preds[name] = p
    df = pd.DataFrame({n: {k: f"{m:.4f} ± {s:.4f}" for k, (m, s) in r.items()} for n, r in res.items()}).T
    if baselines:
        b, bp, true_b = vol_baselines(list(named.values())[0], return_preds=True)
        bdf = pd.DataFrame({n: {k: f"{v:.4f}" for k, v in mt.items()} for n, mt in b.items()}).T
        df = pd.concat([bdf, df])
        preds = {**bp, **preds}
        if true_m is not None and len(true_m) != len(true_b):
            print(f"!! test-set mismatch: model {len(true_m)} rows vs baselines {len(true_b)} (cross_market calendar?) — DM will skip unequal pairs")
        true_m = true_b if true_m is None else true_m
    if tag is not None: PREDS[tag], TRUES[tag] = preds, np.asarray(true_m, float)
    return df


## VIX-core vs. VXN-core, and the decisive DM pairs

In [9]:
QUICK_TEST = False
EPOCHS = 3 if QUICK_TEST else 100
SEEDS  = (1,) if QUICK_TEST else (1, 2, 3)
PAT    = 0 if QUICK_TEST else 15
print(f"QUICK_TEST={QUICK_TEST} -> epochs={EPOCHS}, seeds={SEEDS}, patience={PAT}, device={DEVICE}")

path = f"{INDEX}_vol.csv"
tables_purged = {}
for iv in ("VIX", "VXN"):
    print("\n" + "=" * 70 + f"\n### PURGED {INDEX} \u2014 core with {iv}\n" + "=" * 70)
    tables_purged[iv] = run_ablation({
        f"Hybrid ({iv} core)": replace(Config(), data_path=path, iv_col=iv, hybrid=True, weight_decay=1e-3,
                                       epochs=EPOCHS, patience=PAT),
    }, seeds=SEEDS, tag=f"{iv}-core-purged")
    display(tables_purged[iv])


QUICK_TEST=False -> epochs=100, seeds=(1, 2, 3), patience=15, device=cuda

### PURGED IXIC — core with VIX
[run] Hybrid (VIX core)
purge[IXIC_vol.csv, h=5]: dropped 5 train-boundary + 5 val-boundary rows | train=2486 val=541 test=546 -- zero target-window overlap OK


,RMSE,MAE,QLIKE,R2,Corr
persistence (RW),0.5596,0.4224,1.0006,-0.1805,0.4099
EWMA (l=.94),0.5273,0.4019,0.4939,-0.0482,0.3369
"GARCH(1,1)",0.4868,0.3679,0.4311,0.1069,0.4531
HAR-RV,0.4701,0.3558,0.6250,0.1668,0.4374
VIX-only,0.4723,0.3741,0.6613,0.1592,0.4649
core OLS (HAR-X+VIX+GARCH),0.4605,0.3614,0.6092,0.2005,0.4937
Hybrid (VIX core),0.4688 ± 0.0101,0.3622 ± 0.0079,0.4012 ± 0.0125,0.1712 ± 0.0361,0.5111 ± 0.0066



### PURGED IXIC — core with VXN
[run] Hybrid (VXN core)
purge[IXIC_vol.csv, h=5]: dropped 5 train-boundary + 5 val-boundary rows | train=2486 val=541 test=546 -- zero target-window overlap OK


,RMSE,MAE,QLIKE,R2,Corr
persistence (RW),0.5596,0.4224,1.0006,-0.1805,0.4099
EWMA (l=.94),0.5273,0.4019,0.4939,-0.0482,0.3369
"GARCH(1,1)",0.4868,0.3679,0.4311,0.1069,0.4531
HAR-RV,0.4701,0.3558,0.6250,0.1668,0.4374
VXN-only,0.4543,0.3507,0.5626,0.2222,0.4847
core OLS (HAR-X+VXN+GARCH),0.4498,0.3454,0.5419,0.2375,0.4985
Hybrid (VXN core),0.4510 ± 0.0041,0.3462 ± 0.0030,0.4282 ± 0.0190,0.2332 ± 0.0140,0.5141 ± 0.0017


### DM (purged)


In [10]:
from scipy import stats as _st

def _daily_qlike(pred, true): r = 2 * (true - pred); return np.exp(r) - r - 1
def _daily_se(pred, true):    return (pred - true) ** 2

def dm_test(lossA, lossB, h):
    d = np.asarray(lossA, float) - np.asarray(lossB, float)
    n = len(d); dbar = d.mean(); dc = d - dbar
    g = [float(np.dot(dc[k:], dc[:n - k])) / n for k in range(max(h - 1, 0) + 1)]
    var = (g[0] + 2 * sum(g[1:])) / n
    if var <= 0: return float("nan"), float("nan")
    dm = dbar / np.sqrt(var) * np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)   # HLN
    return dm, 2 * _st.t.sf(abs(dm), df=n - 1)

assert np.allclose(TRUES["VIX-core-purged"], TRUES["VXN-core-purged"]), "test sets differ between the two runs!"
P = {**PREDS["VIX-core-purged"], **PREDS["VXN-core-purged"]}
true = TRUES["VIX-core-purged"]

COMPARISONS = [
    ("Hybrid (VXN core)", "VXN-only"),
    ("Hybrid (VXN core)", "core OLS (HAR-X+VXN+GARCH)"),
    ("Hybrid (VXN core)", "Hybrid (VIX core)"),
    ("Hybrid (VIX core)",  "VXN-only"),
    ("VXN-only",           "VIX-only"),
    ("core OLS (HAR-X+VXN+GARCH)", "core OLS (HAR-X+VIX+GARCH)"),
]

H = Config().target_horizon
rows = []
for a, b in COMPARISONS:
    if a not in P or b not in P:
        rows.append({"A": a, "B": b, "note": "missing"}); continue
    qa, qb = _daily_qlike(P[a], true), _daily_qlike(P[b], true)
    sa, sb = _daily_se(P[a], true),   _daily_se(P[b], true)
    dm_q, p_q = dm_test(qa, qb, H); dm_s, p_s = dm_test(sa, sb, H)
    verdict = ("A better" if dm_q < 0 else "B better") if (p_q == p_q and p_q < 0.05) else "\u2248 (n.s.)"
    rows.append({"A": a, "B": b, "QLIKE A": round(float(qa.mean()), 4), "QLIKE B": round(float(qb.mean()), 4),
                 "DM(QLIKE)": round(dm_q, 2), "p(QLIKE)": round(p_q, 4),
                 "DM(SE)": round(dm_s, 2), "p(SE)": round(p_s, 4), "verdict": verdict})
dm_table_purged = pd.DataFrame(rows)
dm_table_purged


,A,B,QLIKE A,QLIKE B,DM(QLIKE),p(QLIKE),DM(SE),p(SE),verdict
0,Hybrid (VXN core),VXN-only,0.4246,0.5626,-1.99,0.0473,-0.32,0.7470,A better
1,Hybrid (VXN core),core OLS (HAR-X+VXN+GARCH),0.4246,0.5419,-2.32,0.0205,-0.02,0.9804,A better
2,Hybrid (VXN core),Hybrid (VIX core),0.4246,0.3965,0.66,0.5113,-1.65,0.1005,≈ (n.s.)
3,Hybrid (VIX core),VXN-only,0.3965,0.5626,-1.54,0.1235,0.57,0.5678,≈ (n.s.)
4,VXN-only,VIX-only,0.5626,0.6613,-1.74,0.0816,-1.39,0.1666,≈ (n.s.)
5,core OLS (HAR-X+VXN+GARCH),core OLS (HAR-X+VIX+GARCH),0.5419,0.6092,-1.52,0.1280,-1.07,0.2837,≈ (n.s.)
